In [1]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    if len(files) > 0:
        print(root, "->", len(files), "files")
        break

/kaggle/input/datasets/bulentsiyah/plantvillage -> 2 files


In [12]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
GPU: []


2026-08-16 12:49:40.841396: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [7]:
for root, dirs, files in os.walk("/kaggle/input/datasets/bulentsiyah/plantvillage"):
    level = root.replace("/kaggle/input/datasets/bulentsiyah/plantvillage", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 3:
        for f in files[:3]:  # sirf pehli 3 files dikhao har folder ki
            print(f"{indent}  {f}")

plantvillage/
  model_weights.h5
  model_architecture.json
  PlantVillage_resize_224/
    PlantVillage_resize_224/
      s32/
      s26/
      s20/
      s18/
      s25/
      s24/
      s14/
      s27/
      s33/
      s1/
      s11/
      s31/
      s35/
      s13/
      s19/
      s29/
      s37/
      s10/
      s8/
      s5/
      s7/
      s28/
      s9/
      s15/
      s21/
      s2/
      s6/
      s30/
      s3/
      s23/
      s4/
      s16/
      s38/
      s17/
      s36/
      s22/
      s34/
      s12/
  plantvillage_resize_224/
    PlantVillage_resize_224/
      s32/
      s26/
      s20/
      s18/
      s25/
      s24/
      s14/
      s27/
      s33/
      s1/
      s11/
      s31/
      s35/
      s13/
      s19/
      s29/
      s37/
      s10/
      s8/
      s5/
      s7/
      s28/
      s9/
      s15/
      s21/
      s2/
      s6/
      s30/
      s3/
      s23/
      s4/
      s16/
      s38/
      s17/
      s36/
      s22/
      s34/
      s12/


In [8]:
DATASET_PATH = "/kaggle/input/datasets/bulentsiyah/plantvillage/PlantVillage_resize_224/PlantVillage_resize_224"

print("Path exists:", os.path.exists(DATASET_PATH))
print("Classes found:", len(os.listdir(DATASET_PATH)))

Path exists: True
Classes found: 38


In [9]:
classes = sorted(os.listdir(DATASET_PATH), key=lambda x: int(x[1:]))  # s1, s2...s38 ke hisaab se sort
print("Number of classes:", len(classes))
print(classes)

Number of classes: 38
['s1', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's20', 's21', 's22', 's23', 's24', 's25', 's26', 's27', 's28', 's29', 's30', 's31', 's32', 's33', 's34', 's35', 's36', 's37', 's38']


In [10]:
def count_images(folder):
    count = 0
    for root, dirs, files in os.walk(folder):
        for file in files:
            if file.lower().endswith((".jpg", ".jpeg", ".png")):
                count += 1
    return count

print("Total images:", count_images(DATASET_PATH))

# sample ek class ke andar kitni images hain
sample_class = classes[0]
print(f"{sample_class} folder me:", len(os.listdir(os.path.join(DATASET_PATH, sample_class))), "images")

Total images: 1900
s1 folder me: 50 images


In [13]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    color_mode="rgb"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    color_mode="rgb"
)

NUM_CLASSES = len(train_ds.class_names)
print("Classes:", NUM_CLASSES)
print("Class names:", train_ds.class_names)

Found 1900 files belonging to 38 classes.
Using 1520 files for training.
Found 1900 files belonging to 38 classes.
Using 380 files for validation.
Classes: 38
Class names: ['s1', 's10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's2', 's20', 's21', 's22', 's23', 's24', 's25', 's26', 's27', 's28', 's29', 's3', 's30', 's31', 's32', 's33', 's34', 's35', 's36', 's37', 's38', 's4', 's5', 's6', 's7', 's8', 's9']


In [14]:
images, labels = next(iter(train_ds))
print("Image shape:", images.shape)
print("Label shape:", labels.shape)
print("Pixel range:", images.numpy().min(), "to", images.numpy().max())

Image shape: (32, 224, 224, 3)
Label shape: (32, 38)
Pixel range: 0.0 to 255.0


In [15]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
print("Dataset optimization complete.")

Dataset optimization complete.


In [16]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name="data_augmentation")
print("Data augmentation ready.")

Data augmentation ready.


In [17]:
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)
base_model.trainable = False   # pehle freeze karo

inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)
model.summary()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 38)             │        48,678 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,098,249 (15.63 MB)

 Trainable params: 48,678 (190.15 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [18]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
print("Model compiled successfully.")

Model compiled successfully.


In [19]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "/kaggle/working/best_plant_disease_model.keras",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]
print("Callbacks ready.")

Callbacks ready.


In [20]:
EPOCHS = 15
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

Epoch 1/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.1527 - loss: 3.3504
Epoch 1: val_accuracy improved from None to 0.54737, saving model to /kaggle/working/best_plant_disease_model.keras

Epoch 1: finished saving model to /kaggle/working/best_plant_disease_model.keras
48/48 ━━━━━━━━━━━━━━━━━━━━ 100s 2s/step - accuracy: 0.2678 - loss: 2.9334 - val_accuracy: 0.5474 - val_loss: 2.1327 - learning_rate: 0.0010
Epoch 2/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5793 - loss: 1.9200
Epoch 2: val_accuracy improved from 0.54737 to 0.68158, saving model to /kaggle/working/best_plant_disease_model.keras

Epoch 2: finished saving model to /kaggle/working/best_plant_disease_model.keras
48/48 ━━━━━━━━━━━━━━━━━━━━ 77s 2s/step - accuracy: 0.6072 - loss: 1.7737 - val_accuracy: 0.6816 - val_loss: 1.4570 - learning_rate: 0.0010
Epoch 3/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7380 - loss: 1.2872
Epoch 3: val_accuracy improved from 0.68158 to 0.73947, saving model to /

In [21]:
loss, accuracy = model.evaluate(val_ds)
print(f"Final Validation Accuracy: {accuracy*100:.2f}%")
print(f"Final Validation Loss: {loss:.4f}")

12/12 ━━━━━━━━━━━━━━━━━━━━ 14s 1s/step - accuracy: 0.8553 - loss: 0.5397
Final Validation Accuracy: 85.53%
Final Validation Loss: 0.5397


In [22]:
model.save("/kaggle/working/plant_disease_final.keras")
print("Model saved!")

Model saved!


In [23]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # size reduce karega
tflite_model = converter.convert()

with open("/kaggle/working/plant_disease_model.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model saved!")

import os
size_mb = os.path.getsize("/kaggle/working/plant_disease_model.tflite") / (1024*1024)
print(f"TFLite model size: {size_mb:.2f} MB")

INFO:tensorflow:Assets written to: /tmp/tmp6zbpkg1_/assets


INFO:tensorflow:Assets written to: /tmp/tmp6zbpkg1_/assets


Saved artifact at '/tmp/tmp6zbpkg1_'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_238')
Output Type:
  TensorSpec(shape=(None, 38), dtype=tf.float32, name=None)
Captures:
  136390342716752: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  136390342716944: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  136390760328144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136390760332176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136390760331792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136390760332944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136390760330640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136390760330832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136390760330256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136390760332368: TensorSpec(shape=(), dtype=tf.resource, name

W0000 00:00:1786886051.281737      58 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1786886051.281778      58 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1786886051.538018      58 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled


TFLite model saved!
TFLite model size: 4.38 MB


In [25]:
import json

# yahi order hai jo Step 5 me print hua tha (alphabetical string sort)
class_names = sorted(os.listdir(DATASET_PATH))

class_mapping = {i: name for i, name in enumerate(class_names)}

with open("/kaggle/working/class_mapping.json", "w") as f:
    json.dump(class_mapping, f, indent=2)

print("Total classes:", len(class_names))
print("Class mapping saved:", class_mapping)

Total classes: 38
Class mapping saved: {0: 's1', 1: 's10', 2: 's11', 3: 's12', 4: 's13', 5: 's14', 6: 's15', 7: 's16', 8: 's17', 9: 's18', 10: 's19', 11: 's2', 12: 's20', 13: 's21', 14: 's22', 15: 's23', 16: 's24', 17: 's25', 18: 's26', 19: 's27', 20: 's28', 21: 's29', 22: 's3', 23: 's30', 24: 's31', 25: 's32', 26: 's33', 27: 's34', 28: 's35', 29: 's36', 30: 's37', 31: 's38', 32: 's4', 33: 's5', 34: 's6', 35: 's7', 36: 's8', 37: 's9'}


In [26]:
import numpy as np

# ek batch val_ds se lo
for images, labels in val_ds.take(1):
    predictions = model.predict(images)
    
    for i in range(5):  # pehli 5 images test karo
        true_class = class_names[np.argmax(labels[i])]
        pred_class = class_names[np.argmax(predictions[i])]
        confidence = np.max(predictions[i]) * 100
        print(f"True: {true_class} | Predicted: {pred_class} | Confidence: {confidence:.1f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
True: s6 | Predicted: s6 | Confidence: 99.3%
True: s4 | Predicted: s4 | Confidence: 91.3%
True: s11 | Predicted: s11 | Confidence: 65.8%
True: s25 | Predicted: s25 | Confidence: 97.8%
True: s21 | Predicted: s21 | Confidence: 94.7%


In [29]:
import shutil, os

# ek folder banao jisme sab kuch daal denge
EXPORT_DIR = "/kaggle/working/model_export"
os.makedirs(EXPORT_DIR, exist_ok=True)

files_to_copy = [
    "/kaggle/working/best_plant_disease_model.keras",
    "/kaggle/working/plant_disease_final.keras",
    "/kaggle/working/plant_disease_model.tflite",
    "/kaggle/working/class_mapping.json",
]

for f in files_to_copy:
    if os.path.exists(f):
        shutil.copy(f, EXPORT_DIR)
        print("Copied:", f)
    else:
        print("Missing:", f)

print("\nAll files ready in:", EXPORT_DIR)

Copied: /kaggle/working/best_plant_disease_model.keras
Copied: /kaggle/working/plant_disease_final.keras
Copied: /kaggle/working/plant_disease_model.tflite
Copied: /kaggle/working/class_mapping.json

All files ready in: /kaggle/working/model_export


In [30]:
shutil.make_archive("/kaggle/working/plant_disease_model_package", 'zip', EXPORT_DIR)
print("Zip file ready: /kaggle/working/plant_disease_model_package.zip")

Zip file ready: /kaggle/working/plant_disease_model_package.zip
